# SageMaker AI MLflow — Medical Triage Agent with Strands Agents

Welcome to the hands-on SageMaker AI MLflow workshop lab! This notebook will guide you through building an intelligent **Medical Triage Agent** using the [Strands Agents SDK](https://github.com/strands-agents/sdk-python) with a Small Language Model (SLM) deployed on a SageMaker AI endpoint.

**What you will learn:**
- Deploy a Small Language Model (Qwen3-8B) on a SageMaker AI real-time endpoint using JumpStart
- Enable DataCapture on the endpoint to log all request/response pairs to S3
- Build a medical triage agent using Strands Agents SDK with custom tools
- Enable MLflow auto-tracing for Strands agents to capture agent reasoning and tool calls
- Observe agent traces in the SageMaker AI MLflow UI
- Verify DataCapture files in S3 for downstream monitoring and evaluation

### Scenario Overview

You are building a **Medical Triage Agent** that assists healthcare staff by analyzing patient symptoms and recommending appropriate treatment protocols. The agent uses two tools:

1. **`symptom_lookup`** — Given a patient ID, retrieves the patient's reported symptoms from the records database
2. **`treatment_lookup`** — Given a set of symptoms, retrieves the matching condition diagnosis and step-by-step treatment protocol

The agent receives a patient ID, looks up their symptoms, identifies the likely condition, and returns the recommended triage level and treatment steps — all traced automatically in MLflow.

## 1. Deploy a Small Language Model (SLM) on SageMaker AI with DataCapture

We will deploy **Qwen3-8B** using SageMaker JumpStart. This is an 8-billion parameter reasoning model that supports function calling — which is required for Strands agent tool use.

We also enable **DataCapture** on the endpoint. DataCapture automatically records every request and response as `.jsonl` files in S3. These files can then be processed by an automated pipeline (EventBridge → Step Functions → Lambda) to log traces to MLflow and run GenAI evaluations.

> **Note:** Endpoint deployment takes approximately 10-15 minutes. The model requires a GPU instance (ml.g5.2xlarge recommended).

Let's start by setting up our environment 

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
!pip install -U sagemaker==2.253.1 datasets==4.4.1 mlflow==3.9.0 fsspec==2023.9.2 --quiet

# restart kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True) #automatically restarts kernel

In [ ]:
import json
import boto3
import sagemaker
from sagemaker.jumpstart.model import JumpStartModel
from sagemaker.model_monitor import DataCaptureConfig

role = sagemaker.get_execution_role()
sess = sagemaker.Session()

model_id = "huggingface-reasoning-qwen3-8b"
instance_type = "ml.g5.2xlarge"
endpoint_name = "workshop-qwen3-8b"

print(f"Model: {model_id}")
print(f"Instance: {instance_type}")
print(f"Endpoint: {endpoint_name}")

### Configure DataCapture

DataCapture records every request and response sent to the endpoint as `.jsonl` files in S3. This enables downstream automated monitoring and evaluation pipelines.

In [ ]:
# Configure DataCapture to log all request/response pairs to S3
# DataCapture must write to the Studio S3 bucket (created by CloudFormation)
# so that EventBridge can detect new .jsonl files and trigger the monitoring pipeline.
s3_client_tmp = boto3.client('s3')
account_id = boto3.client('sts').get_caller_identity()['Account']
studio_bucket = None
for b in s3_client_tmp.list_buckets()['Buckets']:
    if b['Name'].startswith(f'sagemaker-studio-{account_id}'):
        studio_bucket = b['Name']
        break
assert studio_bucket, 'Studio S3 bucket not found. Ensure the CloudFormation stack deployed successfully.'
print(f'Studio S3 bucket: {studio_bucket}')

data_capture_s3_key = f"data-capture/{endpoint_name}"

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f"s3://{studio_bucket}/{data_capture_s3_key}",
    capture_options=["REQUEST", "RESPONSE"],
    json_content_types=["application/json"],
)

print(f"DataCapture destination: {data_capture_config.destination_s3_uri}")

### Deploy the Model with DataCapture Enabled

In [ ]:
model = JumpStartModel(
    model_id=model_id,
    role=role,
    sagemaker_session=sess,
    env={
        "OPTION_TENSOR_PARALLEL_DEGREE": "1",
        "OPTION_MAX_MODEL_LEN": "12288",
        "OPTION_GPU_MEMORY_UTILIZATION": "0.95",
        "OPTION_ENABLE_AUTO_TOOL_CHOICE": "true",
        "OPTION_TOOL_CALL_PARSER": "hermes",
    },
)

print(f"Deploying {model_id} on {instance_type}...")
print("This will take approximately 10-15 minutes.\n")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=instance_type,
    endpoint_name=endpoint_name,
    accept_eula=True,
    data_capture_config=data_capture_config,
)

print(f"\nEndpoint '{endpoint_name}' is now InService with DataCapture enabled!")

### Quick Test — Verify the Endpoint

Let's send a simple prompt to verify the endpoint is working before we build the agent.

In [ ]:
payload = {
    "messages": [{"role": "user", "content": "Hello! Can you briefly describe what medical triage is? /no_think"}],
    "max_tokens": 200,
    "temperature": 0.3,
}

response = predictor.predict(payload)
print(response["choices"][0]["message"]["content"])

## 2. Configure SageMaker AI MLflow

We will configure MLflow integration with Amazon SageMaker AI for tracking and tracing our agent-based workflows.

In [ ]:
# Retrieve values stored from previous labs
%store -r

%store
if MLFLOW_TRACKING_URI is None or MLFLOW_TRACKING_URI == '':
    print("MLFLOW_TRACKING_URI not found. Please enter your MLflow App ARN.")
MLFLOW_TRACKING_URI

In [ ]:
import mlflow

experiment_name = "medical-triage-agent"

# Set MLflow SDK to your configured MLflow App
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment(experiment_name)

print(f"MLflow version: {mlflow.__version__}")
print(f"Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {experiment_name}")

### Enable Strands Auto-Tracing

`mlflow.strands.autolog()` automatically captures traces for every Strands agent invocation — including tool calls, model reasoning, and final responses — and logs them to your MLflow experiment.

In [ ]:
mlflow.strands.autolog()

## 3. Install Strands Agents SDK and Dependencies

In [ ]:
!pip install strands-agents "strands-agents[sagemaker]" strands-agents-tools --upgrade --quiet
!pip install mypy-boto3-sagemaker-runtime --quiet

## 4. Load Patient Data and Treatment Protocols

We use synthetic patient records and treatment protocols to simulate a medical triage system. Each patient has a unique ID, reported symptoms, age, and severity level.

In [ ]:
from data.data import patient_records, patient_ids
from data.solution_book import treatment_protocols

print(f"Loaded {len(patient_records)} patient records")
print(f"Loaded {len(treatment_protocols)} treatment protocols")

Let's examine the patient records. Each record contains a patient ID, their symptoms, age, and an initial severity assessment.

In [ ]:
patient_records

The treatment protocols map symptom combinations to conditions, triage levels, and step-by-step treatment plans.

In [ ]:
# Show one example protocol
example_symptoms = "chest pain, shortness of breath, sweating"
protocol = treatment_protocols[example_symptoms]
print(f"Symptoms: {example_symptoms}")
print(f"Condition: {protocol['condition']}")
print(f"Triage Level: {protocol['triage_level']}")
print(f"Protocol:")
for step in protocol['protocol']:
    print(f"  {step}")

## 5. Define Agent Tools

In Strands Agents, tools are Python functions decorated with `@tool` that the agent can invoke during its reasoning loop. We define two tools for our medical triage workflow.

In [ ]:
from strands import Agent, tool

### The `symptom_lookup` Tool

This tool allows the agent to look up a patient's symptoms by their patient ID. It returns the symptoms, age, and initial severity assessment.

In [ ]:
@tool()
def symptom_lookup(patient_id: str) -> str:
    """Look up patient symptoms by patient ID.

    Args:
        patient_id: The patient identifier (e.g., PT-001)

    Returns:
        A string with the patient's symptoms, age, and severity level.
    """
    patient_id = patient_id.upper().strip()
    for patient in patient_records:
        if patient['id'] == patient_id:
            return (
                f"Patient {patient['id']}: "
                f"Symptoms: {patient['symptoms']}. "
                f"Age: {patient['age']}. "
                f"Initial severity: {patient['severity']}."
            )
    return f"No patient found with ID: {patient_id}. Valid IDs range from PT-001 to PT-010." 

### The `treatment_lookup` Tool

After identifying the patient's symptoms, the agent uses this tool to retrieve the matching condition diagnosis and treatment protocol.

In [ ]:
@tool()
def treatment_lookup(symptoms: str) -> str:
    """Retrieve treatment protocol based on patient symptoms.

    Args:
        symptoms: The patient's symptoms as a comma-separated string

    Returns:
        A string with the condition, triage level, and treatment steps.
    """
    symptoms_clean = symptoms.lower().strip()
    for key, protocol in treatment_protocols.items():
        if key.lower() == symptoms_clean:
            steps = "\n".join(protocol['protocol'])
            return (
                f"Condition: {protocol['condition']}\n"
                f"Triage Level: {protocol['triage_level']}\n"
                f"Treatment Protocol:\n{steps}"
            )
    return f"No matching treatment protocol found for symptoms: {symptoms}. Please consult a physician." 

## 6. Create the Medical Triage Agent

Now we create the Strands agent using `SageMakerAIModel` as the model provider. Qwen3-8B returns tool calls in the standard OpenAI-compatible format, so no custom parsing is needed.

> **Note:** We disable streaming and use `/no_think` in the system prompt to get direct responses without reasoning traces, which keeps the agent loop clean.

In [ ]:
from strands.models.sagemaker import SageMakerAIModel
import boto3

# Create the SageMaker AI Model object
sagemaker_model = SageMakerAIModel(
    endpoint_config={
        "endpoint_name": endpoint_name,
    },
    payload_config={
        "max_tokens": 1024 * 5,
        "temperature": 0.3,
        "stream": False,
    },
    boto_session=boto3.Session(region_name=sess.boto_region_name),
)

print("SageMakerAIModel created successfully!")

In [ ]:
from strands import Agent

SYSTEM_PROMPT = """You are an expert medical triage assistant. /no_think

You help healthcare staff assess patients and recommend treatment protocols.

You have access to two tools:
1. symptom_lookup: Use this to retrieve a patient's symptoms by their patient ID
2. treatment_lookup: Use this to find the treatment protocol for a given set of symptoms

When given a patient ID:
1. First use symptom_lookup to get the patient's symptoms
2. Then use treatment_lookup with the EXACT symptom string returned by symptom_lookup (the part after 'Symptoms: ' and before the period)
3. Present the findings clearly: patient info, condition, triage level, and treatment steps

Always use the tools in sequence. Do not guess symptoms or treatments."""

# Create the agent
agent = Agent(
    model=sagemaker_model,
    tools=[symptom_lookup, treatment_lookup],
    system_prompt=SYSTEM_PROMPT,
)

print("Medical Triage Agent created successfully!")

## 7. Test the Agent

Let's test our medical triage agent with a sample patient query. The agent should:
1. Use `symptom_lookup` to retrieve the patient's symptoms
2. Use `treatment_lookup` to find the matching treatment protocol
3. Present a clear triage assessment

In [ ]:
result = agent("Can you help me triage patient PT-001?")

Let's test with another patient to see a different triage scenario.

In [ ]:
result = agent("Please assess patient PT-003 and provide the treatment protocol.")

Now let's test with a low-severity case to see how the agent handles non-emergency situations.

In [ ]:
result = agent("What is the triage assessment for patient PT-007?")

## 8. View Agent Traces in MLflow

Now you can view the detailed agent traces in the SageMaker AI MLflow UI.

1. Open your SageMaker AI MLflow UI
2. In the MLflow UI, navigate to the **Experiments** section
3. Select the **medical-triage-agent** experiment
4. Click on the **Traces** tab to see all captured traces

Each trace shows:
- The full agent reasoning loop
- Tool invocations with input arguments and return values
- Model calls with prompts and responses
- Timing information for each step

> **Tip:** Click on individual traces to see the detailed span tree, which shows exactly how the agent reasoned through each step.

## 9. Verify DataCapture Files in S3

Since we enabled DataCapture on the endpoint, every request and response from the agent's calls to the SageMaker endpoint has been captured as `.jsonl` files in S3.

> **Note:** It may take 1-2 minutes after the agent invocations for the DataCapture files to appear in S3. If you don't see any files, wait a moment and re-run the cell below.

In [ ]:
import time

s3_client = boto3.client("s3")

print(f"Checking for DataCapture files in s3://{studio_bucket}/{data_capture_s3_key}/...")

response = s3_client.list_objects_v2(
    Bucket=studio_bucket,
    Prefix=data_capture_s3_key,
    MaxKeys=50,
)

if "Contents" in response:
    jsonl_files = [obj["Key"] for obj in response["Contents"] if obj["Key"].endswith(".jsonl")]
    print(f"Found {len(jsonl_files)} DataCapture .jsonl file(s):")
    for f in jsonl_files:
        print(f"  s3://{studio_bucket}/{f}")
else:
    print("No DataCapture files found yet. Wait a moment and re-run this cell.")

These DataCapture files contain the raw request/response payloads for every inference call made to the endpoint. An automated monitoring pipeline (deployed as part of the workshop CloudFormation stack) processes these files:

1. **Amazon EventBridge** detects new `.jsonl` files in S3
2. **AWS Step Functions** orchestrates the processing workflow
3. A **Lambda function** parses the capture data, logs traces to **MLflow**, and runs basic **GenAI evaluations** (Safety, Relevance, Fluency, Guidelines) using Amazon Bedrock

This means your agent traces are already being logged to MLflow automatically with evaluation scores attached. In the next lab, you will use these agent traces to run more comprehensive LLM-as-a-Judge evaluations including domain-specific guidelines, coherence scoring, bias detection, and human feedback.

## 10. Cleanup

When you are done with the workshop, uncomment and run the cell below to delete the endpoint and avoid ongoing charges.

> **Important:** Do not delete the endpoint if you plan to continue with the next lab (Online Evaluation).

In [ ]:
# Uncomment the lines below to delete the endpoint and model
# predictor.delete_model()
# predictor.delete_endpoint()
# print(f"Endpoint '{endpoint_name}' deleted successfully.")